# Checkpoint 3 — Person 1: never-exit trajectories

Runs DeepSeek-R1-Distill-Qwen-1.5B on the first 100 GSM8K and first 100 MATH-500 questions with DEER's loop,
but with a threshold that can never be reached (τ = 1.1). The model reasons to the end while a trial answer and
its confidence are logged at **every** "Wait" checkpoint (up to 50).

This one run lets us replay DEER at any threshold offline (the τ sweep), check the simulator against live DEER,
and train/test the adaptive controller on real data.

**How to run:** T4 runtime → *Runtime → Run all*. Expected time ~1.5–2 h, most of it in step 6.
Keep the tab open. If Colab disconnects, reconnect and *Run all* again.

## 1. GPU check
*Runtime → Change runtime type → T4 GPU* first. This should print `Tesla T4`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive backup
Outputs are copied to Drive after every run, so a Colab disconnect loses nothing.
If you reconnect, just run all cells again: finished runs are restored from Drive and skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Get the code

In [ ]:
REPO = "https://github.com/PeramVishnuSree/Adaptive-Early-Exit-for-Efficient-Reasoning-Reproducing-and-Extending-DEER.git"
!git clone -q $REPO /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo/Vishnu-evaluation
!git log --oneline -1

## 4. Install (~3–5 min)
vLLM 0.26.0 matches Colab's Python 3.13 / torch 2.11. Exit logic and decoding settings are the official DEER ones.

In [ ]:
!pip install -q "vllm==0.26.0" nltk
!pip install -q "antlr4-python3-runtime==4.11.0" "latex2sympy2_extended[antlr4_11_0]" regex word2number

Check the install. Expect `vllm 0.26.0 | torch 2.11.0+... | cuda True`, then a few generated words.
The first model load takes a few minutes.

In [ ]:
!python -c "import vllm, torch, sympy; print('vllm', vllm.__version__, '| torch', torch.__version__, '| sympy', sympy.__version__, '| cuda', torch.cuda.is_available())"
!python -c "from vllm import LLM, SamplingParams; llm = LLM('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', dtype='float16', max_model_len=4096, gpu_memory_utilization=0.8); print(repr(llm.generate(['What is 2+3?'], SamplingParams(max_tokens=20, temperature=0))[0].outputs[0].text))"

**Only if the check fails with a CUDA error** (`undefined symbol`, `libcudart.so.12`, ...): set `FIX_CUDA = True`,
run this cell, then *Runtime → Restart session*, and run the check cell again.

In [ ]:
FIX_CUDA = False
if FIX_CUDA:
    !pip install -q "torch==2.11.0" "torchvision==0.26.0" "torchaudio==2.11.0" --index-url https://download.pytorch.org/whl/cu129

## 5. Settings
**Do not change these.** Both people must use the same values so results line up question by question.

In [ ]:
import os, glob, json, shutil
%cd /content/repo/Vishnu-evaluation
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
N = 100                 # first 100 questions of each test set
MAX_TOKENS = 8000       # total generation budget (paper uses 16000; reduced to fit the time)
DATASETS = ["gsm8k", "math"]
DTYPE = "float16"       # T4 has no bf16

OUT = "outputs/ckpt"
DRIVE_DIR = "/content/drive/MyDrive/deer_ckpt/person1"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copytree(DRIVE_DIR, OUT, dirs_exist_ok=True)   # restore finished runs after a disconnect

def done(path):
    """A run is finished once its .meta.json exists (written last)."""
    return os.path.exists(path[: -len(".jsonl")] + ".meta.json")

def backup():
    shutil.copytree(OUT, DRIVE_DIR, dirs_exist_ok=True)
    print("backed up to", DRIVE_DIR)

print("already finished:", [os.path.basename(f) for f in sorted(glob.glob(f"{OUT}/*.meta.json"))])

## 6. Never-exit run (slowest step)
GSM8K first, then MATH-500. Each dataset is backed up to Drive as soon as it finishes.

In [ ]:
for ds in DATASETS:
    out = f"{OUT}/never_{ds}.jsonl"
    if done(out):
        print("skip (done):", out); continue
    !python third_party/deer/vllm_deer_logged.py --model_name_or_path $MODEL --dataset_dir data --dataset $ds --limit $N --max_generated_tokens $MAX_TOKENS --think_ratio 0.6 --policy avg1 --dtype $DTYPE --gpu-memory-utilization 0.9 \
        --threshold 1.1 --max_judge_steps 50 --output_file $out 2>&1 | grep -v "Early exit threshold not reached" | tail -8
    backup()

## Score
Grades every run in `outputs/ckpt` with DEER's grader and prints a summary.

In [ ]:
for f in sorted(glob.glob(f"{OUT}/*.jsonl")):
    if f.endswith(".scored.jsonl"):
        continue
    ds = "gsm8k" if f.endswith("_gsm8k.jsonl") else "math"
    !python scripts/score_run.py --input $f --dataset $ds
backup()

rows = []
for f in sorted(glob.glob(f"{OUT}/*.summary.json")):
    s = json.load(open(f))
    rows.append((s["input"], s["n"], 100 * s["accuracy"], s["avg_output_tokens"], s["avg_compute_tokens"], (s.get("wall_time_s") or 0) / 60))
print(f"{'run':40s} {'n':>4s} {'acc%':>6s} {'out tok':>8s} {'compute':>8s} {'min':>6s}")
for r in rows:
    print(f"{r[0]:40s} {r[1]:4d} {r[2]:6.1f} {r[3]:8.0f} {r[4]:8.0f} {r[5]:6.1f}")

## Save and send
Downloads `ckpt_person1_never.zip`. Send it to Vishnu (or share the Drive folder `deer_ckpt/person1`).
Raw outputs are gitignored, so they are shared as a zip, not pushed.

In [ ]:
!zip -qr ckpt_person1_never.zip outputs/ckpt
from google.colab import files
files.download("ckpt_person1_never.zip")